# 1. 셀레니움
셀레니움(Selenium)은 파이썬 코드로 실제 웹 브라우저(Chrome, Edge 등)를 자동으로 제어할 수 있게 해주는 웹 자동화 라이브러리입니다. 일반적인 requests 크롤링은 서버에서 받은 HTML만 분석하지만, Selenium은 브라우저를 직접 실행하여 버튼 클릭, 검색 입력, 스크롤, 로그인, 페이지 이동 등의 사용자 동작을 그대로 수행할 수 있기 때문에 JavaScript로 동적으로 생성되는 데이터까지 가져올 수 있습니다. 따라서 멜론, 유튜브, 쇼핑몰처럼 JavaScript 렌더링이 많은 사이트의 크롤링이나 웹 자동화 테스트에서 매우 많이 사용됩니다. 보통 webdriver.Chrome()으로 브라우저를 실행하고, find_element()로 요소를 찾으며, click(), send_keys() 등을 통해 자동화 작업을 수행합니다.

In [10]:
import requests
from bs4 import BeautifulSoup

url = "http://127.0.0.1:5500/07.html"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")
fruits = soup.select(".fruit")
print(fruits)

[]


In [11]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time


> 👉 find_elements()는 Selenium에서 HTML 요소를 여러 개 찾을 때 사용하는 메서드입니다. 하나의 요소만 찾는 find_element()와 달리, 조건에 맞는 요소들을 리스트 형태로 모두 반환합니다. 예를 들어 여러 개의 이미지, 게시글, 테이블 행(tr) 등을 반복해서 크롤링할 때 매우 자주 사용됩니다. 찾는 방식은 By.ID, By.CLASS_NAME, By.CSS_SELECTOR, By.XPATH 등 다양한 선택자를 사용할 수 있으며, 반환 결과는 리스트이므로 for문으로 반복 처리하는 경우가 많습니다. 또한 find_element()는 요소를 찾지 못하면 에러가 발생하지만, find_elements()는 빈 리스트([])를 반환하기 때문에 반복 크롤링에서 더 안정적으로 사용되는 경우도 많습니다.

In [12]:
driver = webdriver.Chrome()
url = "http://127.0.0.1:5500/07.html"
driver.get(url)

# JavaScript 실행 대기
time.sleep(2)
fruits = driver.find_elements(By.CLASS_NAME, "fruit")

for fruit in fruits:
    print(fruit.text)

driver.quit() # 브라우저 닫기

사과
바나나
오렌지
딸기


# 2. 멜론 아티스트 곡 리스트 크롤링

### ※ xpath
XPath는 XML 또는 HTML 문서 내에서 특정 요소나 속성을 선택하기 위해 사용되는 경로 표현 언어입니다. 웹 크롤링이나 자동화 도구에서 주로 사용되며, 요소를 효율적으로 찾을 수 있도록 도와줍니다. 일반적인 XPath는 특정 위치나 속성을 기준으로 요소를 선택하는 상대적인 경로를 사용합니다. 또한 full xpath는 루트 요소에서 시작하여 대상 요소까지의 절대적인 경로를 나타냅니다. 따라서 문서 구조가 변경되면 경로가 깨질 가능성이 높습니다.

In [3]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [ ]:
# def melon_search_from_main(keyword):
#     options = Options()
#     options.add_argument("--start-maximized")

#     driver = webdriver.Chrome(options=options)
#     wait = WebDriverWait(driver, 10)

#     data = []

#     try:
#         driver.get("https://www.melon.com/")
#         # time.sleep(2)

#         search_box = wait.until(
#             EC.presence_of_element_located((By.ID, "top_search"))
#         )

#         search_box.clear()
#         search_box.send_keys(keyword)
#         search_box.send_keys(Keys.ENTER)

#         # time.sleep(3)

#         song_tab = wait.until(
#             EC.element_to_be_clickable(
#                 (By.XPATH, '//*[@id="divCollection"]/ul/li[3]/a/span')
#             )
#         )
#         song_tab.click()

#         # time.sleep(3) 

#         song_table = wait.until(
#             EC.presence_of_element_located(
#                 (By.XPATH, '//*[@id="frm_defaultList"]/div/table')
#             )
#         )

#         rows = song_table.find_elements(By.CSS_SELECTOR, "tbody tr")

#         print("찾은 행 개수:", len(rows))

#         for row in rows:
#             try:
#                 cols = row.find_elements(By.TAG_NAME, "td")

#                 if len(cols) < 5:
#                     continue

#                 # 곡명 td
#                 title_text = cols[2].text.strip()
#                 title_lines = [
#                     line.strip()
#                     for line in title_text.split("\n")
#                     if line.strip()
#                 ]

#                 title = ""

#                 for line in title_lines:
#                     if (
#                         "재생" not in line
#                         and "담기" not in line
#                         and "상세정보" not in line
#                         and not line.startswith("Title")
#                     ):
#                         title = line
#                         break

#                 # Title 줄이 더 정확한 경우 보정
#                 for line in title_lines:
#                     if line.startswith("Title "):
#                         title = line.replace("Title ", "").strip()
#                         break

#                 # 아티스트
#                 artist_text = cols[3].text.strip()
#                 artist_lines = [
#                     line.strip()
#                     for line in artist_text.split("\n")
#                     if line.strip()
#                 ]
#                 artist = artist_lines[0] if artist_lines else ""

#                 # 앨범
#                 album_text = cols[4].text.strip()
#                 album_lines = [
#                     line.strip()
#                     for line in album_text.split("\n")
#                     if line.strip()
#                 ]
#                 album = album_lines[0] if album_lines else ""

#                 # 좋아요 수
#                 like = ""
#                 try:
#                     like = row.find_element(
#                         By.CSS_SELECTOR,
#                         "button.like span.cnt"
#                     ).text.strip()
#                 except:
#                     pass

#                 if title:
#                     data.append({
#                         "곡명": title,
#                         "아티스트": artist,
#                         "앨범": album,
#                         "좋아요수": like
#                     })

#             except Exception as e:
#                 print("행 처리 오류:", e)

#         df = pd.DataFrame(data)

#         if not df.empty:
#             df.index = df.index + 1

#         file_name = f"melon_{keyword}_songs.csv"
#         df.to_csv(file_name, encoding="utf-8-sig")

#         print(f"CSV 저장 완료: {file_name}")
#         print(f"총 {len(df)}곡 수집 완료")

#         return df

#     finally:
#         # driver.quit()
#         pass

In [ ]:
# def melon_search_from_main(keyword):
#     options = Options()
#     options.add_argument("--start-maximized")

#     driver = webdriver.Chrome(options=options)
#     wait = WebDriverWait(driver, 10)

#     data = []

#     try:
#         driver.get("https://www.melon.com/")

#         search_box = wait.until(
#             EC.presence_of_element_located((By.ID, "top_search"))
#         )

#         search_box.clear()
#         search_box.send_keys(keyword)
#         search_box.send_keys(Keys.ENTER)

#         song_tab = wait.until(
#             EC.element_to_be_clickable(
#                 (By.XPATH, '//*[@id="divCollection"]/ul/li[3]/a/span')
#             )
#         )
#         song_tab.click()

#         song_table = wait.until(
#             EC.presence_of_element_located(
#                 (By.XPATH, '//*[@id="frm_defaultList"]/div/table')
#             )
#         )

#         for i in range(5):1
#             num = 1
#             span.pageNum a  sendPage("1")
#             # --------------------------------
#             rows = song_table.find_elements(By.CSS_SELECTOR, "tbody tr")

#             print("찾은 행 개수:", len(rows))

#             for row in rows:
#                 try:
#                     cols = row.find_elements(By.TAG_NAME, "td")

#                     if len(cols) < 5:
#                         continue

#                     # 곡명 td
#                     title_text = cols[2].text.strip()
#                     title_lines = [
#                         line.strip()
#                         for line in title_text.split("\n")
#                         if line.strip()
#                     ]

#                     title = ""

#                     for line in title_lines:
#                         if (
#                             "재생" not in line
#                             and "담기" not in line
#                             and "상세정보" not in line
#                             and not line.startswith("Title")
#                         ):
#                             title = line
#                             break

#                     # Title 줄이 더 정확한 경우 보정
#                     for line in title_lines:
#                         if line.startswith("Title "):
#                             title = line.replace("Title ", "").strip()
#                             break

#                     # 아티스트
#                     artist_text = cols[3].text.strip()
#                     artist_lines = [
#                         line.strip()
#                         for line in artist_text.split("\n")
#                         if line.strip()
#                     ]
#                     artist = artist_lines[0] if artist_lines else ""

#                     # 앨범
#                     album_text = cols[4].text.strip()
#                     album_lines = [
#                         line.strip()
#                         for line in album_text.split("\n")
#                         if line.strip()
#                     ]
#                     album = album_lines[0] if album_lines else ""

#                     # 좋아요 수
#                     like = ""
#                     try:
#                         like = row.find_element(
#                             By.CSS_SELECTOR,
#                             "button.like span.cnt"
#                         ).text.strip()
#                     except:
#                         pass

#                     if title:
#                         data.append({
#                             "곡명": title,
#                             "아티스트": artist,
#                             "앨범": album,
#                             "좋아요수": like
#                         })
#                 # -------------------------------------------------------------


#                 except Exception as e:
#                     print("행 처리 오류:", e)

#         df = pd.DataFrame(data)

#         if not df.empty:
#             df.index = df.index + 1

#         file_name = f"melon_{keyword}_songs.csv"
#         df.to_csv(file_name, encoding="utf-8-sig")

#         print(f"CSV 저장 완료: {file_name}")
#         print(f"총 {len(df)}곡 수집 완료")

#         return df

#     finally:
#         # driver.quit()
#         pass

In [ ]:
# melon_search_from_main("조용필")

찾은 행 개수: 50
찾은 행 개수: 50
찾은 행 개수: 50
찾은 행 개수: 50
찾은 행 개수: 50
CSV 저장 완료: melon_조용필_songs.csv
총 250곡 수집 완료


,곡명,아티스트,앨범,좋아요수
1,바람의 노래,조용필,조용필 16집,"26,576"
2,꿈,조용필,The Dreams,"16,275"
3,Bounce,조용필,Hello,"40,020"
4,HOT 잊혀진 사랑,조용필,30주년 기념 음반 Part 1,"2,414"
5,HOT 단발머리,조용필,조용필 1집 (Remastered),"11,088"
...,...,...,...,...
246,꿈 (2024 Remastered Ver.),조용필,웰컴투 삼달리 X 조용필 (2024 Remastering),"1,088"
247,사랑은 아직도 끝나지 않았네,조용필,조용필 1집 (Remastered),751
248,못찾겠다 꾀꼬리,조용필,조용필 4집,"2,172"
249,잊혀진 사랑,조용필,조용필 1집 (Remastered),684


In [34]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

In [37]:
def extract_current_page(driver, wait):
    data = []

    try:
        song_table = wait.until(
            EC.presence_of_element_located(
                (By.XPATH, '//*[@id="frm_defaultList"]/div/table')
            )
        )
    except TimeoutException:
        return []

    rows = song_table.find_elements(By.CSS_SELECTOR, "tbody tr")

    for row in rows:
        cols = row.find_elements(By.TAG_NAME, "td")

        if len(cols) < 5:
            continue

        title_lines = [
            line.strip()
            for line in cols[2].text.split("\n")
            if line.strip()
        ]

        title = ""

        for line in title_lines:
            if line.startswith("Title "):
                title = line.replace("Title ", "").strip()
                break

        if not title:
            for line in title_lines:
                if (
                    "재생" not in line
                    and "담기" not in line
                    and "상세정보" not in line
                    and not line.startswith("Title")
                ):
                    title = line
                    break

        artist_lines = [
            line.strip()
            for line in cols[3].text.split("\n")
            if line.strip()
        ]
        artist = artist_lines[0] if artist_lines else ""

        album_lines = [
            line.strip()
            for line in cols[4].text.split("\n")
            if line.strip()
        ]
        album = album_lines[0] if album_lines else ""

        try:
            like = row.find_element(
                By.CSS_SELECTOR,
                "button.like span.cnt"
            ).text.strip()
        except:
            like = ""

        if title:
            data.append({
                "곡명": title,
                "아티스트": artist,
                "앨범": album,
                "좋아요수": like
            })

    return data


In [ ]:
def melon_search_all_pages(keyword, max_page=30):
    options = Options()
    options.add_argument("--start-maximized")

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 10)

    all_data = []

    try:
        driver.get("https://www.melon.com/")
        time.sleep(2)  # sleep 하는 이유는 데이터 불러오는 거 기다리기, 사람처럼 행동(바로바로 찾으면 봇으로 감지)

        search_box = wait.until(
            EC.presence_of_element_located((By.ID, "top_search"))
        )

        search_box.clear()
        search_box.send_keys(keyword)
        search_box.send_keys(Keys.ENTER)

        time.sleep(3)

        song_tab = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, '//*[@id="divCollection"]/ul/li[3]/a/span')
            )
        )

        song_tab.click()
        time.sleep(3)

        for page in range(1, max_page + 1):
            page_data = extract_current_page(driver, wait)

            if not page_data:
                print(f"{page}페이지 데이터가 없어 종료합니다.")
                break

            all_data.extend(page_data)
            print(f"{page}페이지 크롤링 완료: {len(page_data)}곡")

            next_start_index = page * 50 + 1

            try:
                driver.execute_script( # 자바스크립트를 실행할 수 있게 해주는 함수
                    f"pageObj.sendPage('{next_start_index}');"  # window.scrollTo(첫 페이지, 끝 페이지)를 사용해도 됨
                )
                time.sleep(3)

            except Exception as e:
                print("다음 페이지 이동 실패:", e)
                break

        df = pd.DataFrame(all_data)

        if not df.empty:
            df = df.drop_duplicates(
                subset=["곡명", "아티스트", "앨범"]
            )
            df.index = df.index + 1

        file_name = f"melon_{keyword}_all_songs.csv"

        df.to_csv(
            file_name,
            encoding="utf-8-sig"
        )

        print(f"CSV 저장 완료: {file_name}")
        print(f"총 {len(df)}곡 수집 완료")

        return df

    finally:
        driver.quit()

In [41]:
df = melon_search_all_pages("조용필", 30)
df

1페이지 크롤링 완료: 50곡
2페이지 크롤링 완료: 50곡
3페이지 크롤링 완료: 50곡
4페이지 크롤링 완료: 50곡
5페이지 크롤링 완료: 50곡
6페이지 크롤링 완료: 50곡
7페이지 크롤링 완료: 50곡
8페이지 크롤링 완료: 50곡
9페이지 크롤링 완료: 50곡
10페이지 크롤링 완료: 50곡
11페이지 크롤링 완료: 50곡
12페이지 크롤링 완료: 50곡
13페이지 크롤링 완료: 50곡
14페이지 크롤링 완료: 50곡
15페이지 크롤링 완료: 50곡
16페이지 크롤링 완료: 50곡
17페이지 크롤링 완료: 50곡
18페이지 크롤링 완료: 50곡
19페이지 크롤링 완료: 50곡
20페이지 크롤링 완료: 50곡
21페이지 크롤링 완료: 50곡
22페이지 크롤링 완료: 50곡
23페이지 크롤링 완료: 50곡
24페이지 크롤링 완료: 50곡
25페이지 크롤링 완료: 50곡
26페이지 크롤링 완료: 20곡
27페이지 데이터가 없어 종료합니다.
CSV 저장 완료: melon_조용필_all_songs.csv
총 1270곡 수집 완료


,곡명,아티스트,앨범,좋아요수
1,바람의 노래,조용필,조용필 16집,"26,576"
2,꿈,조용필,The Dreams,"16,275"
3,HOT 잊혀진 사랑,조용필,30주년 기념 음반 Part 1,"2,414"
4,Bounce,조용필,Hello,"40,021"
5,HOT 단발머리,조용필,조용필 1집 (Remastered),"11,088"
...,...,...,...,...
1266,님이여,조용필,기쁜 우리 젊은날의 가요 4집,15
1267,마음속의 그림자,조용필,기쁜 우리 젊은날의 가요 4집,14
1268,세월이 가면,조용필,기쁜 우리 젊은날의 가요 1집,24
1269,Bounce - 조용필 (MR),음악상자,음악상자 12집,3


# 3. 스타벅스 서울 전체 매장 크롤링

In [42]:
import time
import re
import pandas as pd

from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from bs4 import BeautifulSoup


In [54]:
def fetch_starbucks():
    url = "https://www.starbucks.co.kr/index.do"

    driver = webdriver.Chrome()
    driver.maximize_window()

    driver.get(url)
    time.sleep(2)

    # # 사과문 팝업 닫기
    # try:
    #     close_btn = WebDriverWait(driver, 5).until(
    #         EC.element_to_be_clickable(
    #             (By.XPATH, "/html/body/div[5]/p/a")
    #         )
    #     )

    #     close_btn.click()
    #     print("사과문 팝업 닫기 완료")
    #     time.sleep(1)

    # except:
    #     print("사과문 팝업이 없거나 이미 닫혀 있습니다.")


    # 메뉴 이동
    action = ActionChains(driver)

    first_tag = driver.find_element(
        By.CSS_SELECTOR,
        "#gnb > div > nav > div > ul > li.gnb_nav03"
    )

    second_tag = driver.find_element(
        By.CSS_SELECTOR,
        "#gnb > div > nav > div > ul > li.gnb_nav03 > div > div > div > ul:nth-child(1) > li:nth-child(3) > a"  # 검사해서 가져옴
    )

    action.move_to_element(first_tag) \
          .move_to_element(second_tag) \
          .click() \
          .perform()

    # 서울 선택
    seoul_tag = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((
            By.CSS_SELECTOR,
            "#container > div > form > fieldset > div > section > article.find_store_cont > article > article:nth-child(4) > div.loca_step1 > div.loca_step1_cont > ul > li:nth-child(1) > a"
        ))
    )

    seoul_tag.click()

    # 구 목록 로딩 대기
    WebDriverWait(driver, 5).until(
        EC.presence_of_all_elements_located(
            (By.CLASS_NAME, "set_gugun_cd_btn")
        )
    )

    gu_elements = driver.find_elements(
        By.CLASS_NAME,
        "set_gugun_cd_btn"
    )

    # 전체 선택
    gu_elements[0].click()

    # 매장 목록 로딩 대기
    WebDriverWait(driver, 5).until(
        EC.presence_of_all_elements_located(
            (By.CLASS_NAME, "quickResultLstCon")
        )
    )

    # HTML 가져오기
    req = driver.page_source

    soup = BeautifulSoup(req, "html.parser")

    stores = soup.find(
        'ul',
        'quickSearchResultBoxSidoGugun'
    ).find_all('li')

    # 데이터 저장 리스트
    store_list = []
    addr_list = []
    lat_list = []
    lng_list = []

    # 데이터 추출
    for store in stores:

        store_name = store.find("strong").text

        store_addr = store.find("p").text

        # 전화번호 제거
        store_addr = re.sub(
            r'\d{4}-\d{4}$',
            '',
            store_addr
        ).strip()

        store_lat = store['data-lat']       # 위도
        store_lng = store['data-long']      # 경도 

        store_list.append(store_name)
        addr_list.append(store_addr)
        lat_list.append(store_lat)
        lng_list.append(store_lng)

    # 데이터프레임 생성
    df = pd.DataFrame({
        'store': store_list,
        'addr': addr_list,
        'lat': lat_list,
        'lng': lng_list
    })

    driver.quit()

    return df

In [55]:
starbucks_df = fetch_starbucks()

# CSV 저장
starbucks_df.to_csv(
    "starbucks_seoul.csv",
    index=False,
    encoding='utf-8-sig'
)

print("데이터가 starbucks_seoul.csv 파일로 저장되었습니다.")
print(starbucks_df.head())

데이터가 starbucks_seoul.csv 파일로 저장되었습니다.
       store                        addr         lat          lng
0  역삼아레나빌딩       서울특별시 강남구 언주로 425 (역삼동)   37.501087   127.043069
1   논현역사거리      서울특별시 강남구 강남대로 538 (논현동)   37.510178   127.022223
2  신사역성일빌딩      서울특별시 강남구 강남대로 584 (논현동)  37.5139309  127.0206057
3   국기원사거리      서울특별시 강남구 테헤란로 125 (역삼동)   37.499517   127.031495
4   대치재경빌딩    서울특별시 강남구 남부순환로 2947 (대치동)   37.494668   127.062583
